In [ ]:
import numpy as np
import pandas as pd 
import missingno as msno
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler



In [2]:
data=pd.read_csv("fifa21 raw data v2.csv")

C:\Users\Mohsin's PC\AppData\Local\Temp\ipykernel_12372\2840900921.py:1: DtypeWarning: Columns (0: Hits) have mixed types. Specify dtype option on import or set low_memory=False.
  data=pd.read_csv("fifa21 raw data v2.csv")


In [3]:
data.shape
data.drop_duplicates(inplace=True)


In [4]:
data.rename(columns= {"↓OVA": "OVA"}, inplace= True)

In [5]:
data['Hits']=data['Hits'].str.replace('K','').astype(float)
data['Hits']=data['Hits'].fillna(data['Hits'].median())

In [6]:
data['Height']=data['Height'].str.replace('cm','')
data['Weight']=data['Weight'].str.replace('lbs','')

data['Weight']=data['Weight'].str.replace('kg','').astype(int)
# data['Value']=data['Value'].str.replace('€','')
# data['Wage']=data['Wage'].str.replace('€','')
# data['Release Clause']=data['Release Clause'].str.replace('€','')
# data['Release Clause']=data['Release Clause'].str.replace('M','000000')
# data['Release Clause']=data['Release Clause'].str.replace('K','000').astype(float)
# data['Value']=data['Value'].str.replace('M','000000')
# data['Value']=data['Value'].str.replace('K','000').astype(float)
# data['Wage']=data['Wage'].str.replace('K','000').astype(float)





In [7]:
def remove1(val1):
    val1=str(val1).replace('€','')
    if val1.endswith('M'):
        return float(val1.replace('M',''))*1000000
    elif val1.endswith('K'):
        return float(val1.replace('K',''))*1000
    else:
       return float(val1)

data['Value']=data['Value'].apply(remove1)
data['Wage']=data['Wage'].apply(remove1)
data['Release Clause']=data['Release Clause'].apply(remove1)

In [8]:
ll=['Club','Nationality','A/W','D/W','W/F','SM','IR']
le=LabelEncoder()
for col in ll:
    data[col]=le.fit_transform(data[col])

In [9]:
data = data.drop('Loan Date End', axis=1, errors='ignore')


In [10]:
data[['Contract start','Contract end']]=data['Contract'].str.split(' ~ ',expand=True)
data = data.drop('Contract', axis=1, errors='ignore')



In [11]:
data['Joined']=pd.to_datetime(data['Joined'])
year=data[2026-data['Joined'].dt.year>10]
print(year['Name'])



0                 L. Messi
2                 J. Oblak
3             K. De Bruyne
5           R. Lewandowski
9            M. ter Stegen
               ...        
18390           J. Stevens
18416        B. Al Bahrani
18781            S. Callan
18915    Chen-Zeng Tailang
18917            Gao Xiang
Name: Name, Length: 1658, dtype: str


In [12]:
total=0
for i in data['Wage']:
    total+=i
avg=total/len(data['Wage'])
avg=avg*0.35
    

In [13]:
iss=data[data['Wage']<=avg]
print(iss['Name'])


289      Welington Dano
292      Juiano Mestres
346             Ismaily
354              Marlos
358              Taison
              ...      
18974            Xia Ao
18975          B. Hough
18976       R. McKinley
18977      Wang Zhen'ao
18978         Zhou Xiao
Name: Name, Length: 10563, dtype: str


In [14]:
data['Join year']=data['Joined'].dt.year
data['Join month']=data['Joined'].dt.month
data['Join day']=data['Joined'].dt.day
data = data.drop('Joined', axis=1, errors='ignore')
print(data['Club'].dtype)
data['Club']=data['Club'].astype(str)



int64


In [15]:
for i in data.columns:
    if data[i].dtype=='Object':
        data[i]=data[i].str.strip("\n")


In [16]:
cat = data.select_dtypes(include=['float64','int64']).columns


In [17]:
min_max_scaler = MinMaxScaler()
for i in cat:
    Q1 = data[i].quantile(0.25)
    Q3 = data[i].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = data[(data[i] < lower_bound) | (data[i] > upper_bound)]
    data[i] = pd.DataFrame(min_max_scaler.fit_transform(data[[i]]))



In [18]:
min_count = data['Preferred Foot'].value_counts().min()

balanced_list = []
for city, group in data.groupby('Preferred Foot'):
    balanced_list.append(group.sample(min_count))
data_balanced = pd.concat(balanced_list).reset_index(drop=True)




In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
data['Preferred Foot'].value_counts().plot(kind='bar', ax=axes[0])
data_balanced['Preferred Foot'].value_counts().plot(kind='bar', ax=axes[1])

plt.show()